# RFT Code Repair: MAI-Code-1.1-Flash (preview)

Teach a coding model to prefer small, functional, regression-safe repairs with **managed reinforcement fine-tuning**. `MAI-Code-1.1-Flash` supports **RFT only**, not SFT, DPO, or interactive training.

We use the supplied **50 training / 15 validation** Python repair tasks and the unchanged `code_repair_quality` grader, powered by `o3-mini-2025-01-31`. The model generates JSON containing a unified-diff `patch`, an `explanation`, and `tests`; the grader judges the response against the reference patch, test expectations, and requirements.

Unlike SFT, there is no stored assistant answer to imitate. Unlike an executable test grader, this score-model grader does not apply the patch or run the tests. Its reward is a code-review judgment, not proof of correctness.

**Execution boundary:** Sections 1-4 are offline. Later cells connect to Foundry, upload data, and submit/recover paid training. Do not use **Run All** as an offline smoke test. This notebook does not provision resources, run calibration or inference, deploy a model, or delete anything.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Prepare the local environment

Follow the [central setup guide](../../../GETTING_STARTED.md), select the **Foundry managed** kernel, and use this demo directory or its `notebooks/` directory as the working directory. These preparation cells do not load credentials or instantiate a cloud client.

The versioned model identifier `MAI-Code-1.1-Flash-2026-09-15` is fixed here and in [demo metadata](../demo.yaml). Confirm that exact version's availability in your project and region; the family name alone is not the training identifier. We do not copy a training configuration from another model's tutorial.

In [ ]:
import csv
import hashlib
import io
import json
import math
import os
import re
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

from shared.auth import create_project_context
from shared.config import load_foundry_config
from shared.dataset_validation import (
    DatasetIssue,
    hash_file,
    validate_jsonl,
    validate_split_isolation,
)
from shared.experiment_manifest import ExperimentManifest, RuntimeManifest
from shared.foundry_operations import (
    create_or_reuse_fine_tuning_job,
    monitor_fine_tuning_job,
    upload_or_reuse_file,
    wait_for_file,
)
from shared.logging import redact_text, safe_reference
from shared.naming import NameFactory
from shared.retry import retry_call


def find_demo_root():
    current = Path.cwd().resolve()
    candidates = [current, current.parent] if current.name == "notebooks" else [current]
    for candidate in candidates:
        if (
            (candidate / "data" / "preserved" / "code-repair-rft-training.jsonl").is_file()
            and (candidate / "demo.yaml").is_file()
        ):
            return candidate
    raise FileNotFoundError("Run from the code-repair demo or its notebooks directory.")


DEMO_ROOT = find_demo_root()
DEMO_ID = "rft-mai-code-repair"
MODEL = "MAI-Code-1.1-Flash-2026-09-15"
TRAINING_TYPE = "globalStandard"
OUTPUT_ROOT = DEMO_ROOT / "outputs"
STATE_PATH = OUTPUT_ROOT / "submission-state.json"
GRADER_PATH = DEMO_ROOT / "graders" / "code_repair_quality.json"
GRADER_FIELDS = ("reference_patch", "test_expectations", "requirements")
FILE_TIMEOUT_SECONDS = 600
MONITOR_TIMEOUT_SECONDS = 21_600
POLL_INTERVAL_SECONDS = 60

print({"model": MODEL, "method": "managed RFT", "offline_preparation": True})

## 2. Validate the exact supplied datasets

Read the [data provenance notes](../data/README.md). The files are preserved byte-for-byte, with Git text conversion disabled. Check schema, secrets, counts, byte sizes, hashes, duplicates, and split isolation **before upload**.

Only `messages` is model input. `reference_patch`, `test_expectations`, and `requirements` are grader metadata; do not append the reference patch to the model prompt. The records intentionally contain no assistant message, so we use an explicit RFT row validator rather than the shared SFT assistant-answer requirement.

In [ ]:
def validate_rft_row(row, line_number):
    if set(row) != {"messages", *GRADER_FIELDS}:
        yield DatasetIssue("error", "rft.keys", "Unexpected code-repair row fields", line_number)
    for field in GRADER_FIELDS:
        value = row.get(field)
        if not isinstance(value, str) or not value.strip():
            yield DatasetIssue("error", "rft.reference", f"{field} must be non-empty text", line_number)
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) != 2:
        yield DatasetIssue("error", "rft.messages", "Expected developer and user messages", line_number)
        return
    for message, role in zip(messages, ("developer", "user")):
        if (
            not isinstance(message, dict)
            or set(message) != {"role", "content"}
            or message.get("role") != role
            or not isinstance(message.get("content"), str)
            or not message["content"].strip()
        ):
            yield DatasetIssue("error", "rft.message", f"Invalid {role} message", line_number)


data_manifest = json.loads((DEMO_ROOT / "data" / "hashes.json").read_text(encoding="utf-8"))
DATASETS = {
    label: {**artifact, "path": DEMO_ROOT / artifact["path"]}
    for label, artifact in zip(("training", "validation"), data_manifest["artifacts"], strict=True)
}
dataset_rows = {}
dataset_hashes = {}
prompt_sets = {}
for label, contract in DATASETS.items():
    report = validate_jsonl(
        contract["path"],
        required_keys=("messages", *GRADER_FIELDS),
        validate_messages=False,
        row_validator=validate_rft_row,
    ).require_valid()
    for key in ("rows", "bytes", "sha256"):
        if getattr(report, key) != contract[key]:
            raise ValueError(f"{label} {key} does not match the preserved data manifest")
    if report.duplicate_rows:
        raise ValueError(f"{label} contains duplicate records")
    rows = [json.loads(line) for line in contract["path"].read_text(encoding="utf-8").splitlines()]
    prompts = {row["messages"][-1]["content"] for row in rows}
    if len(prompts) != len(rows):
        raise ValueError(f"{label} contains duplicate user prompts")
    dataset_rows[label] = rows
    dataset_hashes[label] = report.sha256
    prompt_sets[label] = prompts
    print({"split": label, "rows": report.rows, "bytes": report.bytes, "sha256": report.sha256})

validate_split_isolation({label: contract["path"] for label, contract in DATASETS.items()})
if prompt_sets["training"] & prompt_sets["validation"]:
    raise ValueError("Training and validation user prompts overlap")
print({"duplicate_prompts": 0, "split_overlap": 0})

### What a candidate must repair

For example, the first training task asks the model to fix a score normalizer that treats zero as absent. The reference changes the falsy check to `value is None`, validates a positive divisor, and keeps the upper cap. A candidate must provide a usable patch rather than merely describe the bug.

Some edge-case expectations live in grader metadata rather than the user request. Review prompt/reference alignment and valid alternative solutions before adapting this small teaching dataset. File validation is not a correctness review of every reference patch.

The 15 validation tasks are disjoint from training, but the job uses them during optimization. They are **not an untouched final test set**.

## 3. Load the supplied score-model grader

The [grader definition](../graders/code_repair_quality.json) is unchanged: `o3-mini-2025-01-31`, seed `42`, range `[0, 1]`, and pass threshold `0.7`. Its rubric weights functional correctness `0.55`, requirements/edge cases `0.20`, minimality/regression safety `0.15`, and usable patch/explanation `0.10`.

We verify its hash and bindings offline. `{{item.*}}` references must match the JSONL fields; `{{sample.output_text}}` is the complete candidate response. We leave `response_format` unset: the data prompt asks for JSON, but this grader scores JSON-as-text, not `sample.output_json`.

**Calibration is a separate paid workflow.** Before trusting the reward, compare correct, wrong, malformed, and prompt-injection candidates using an approved grading budget. These local checks do not call the grading model or establish score quality.

In [ ]:
GRADER_SHA256 = "57ec9f07ff12f001f9cdbf8a1aaa9a2e9d5e2e080d9dea1a000e0ae1fc7a64d7"
if hash_file(GRADER_PATH) != GRADER_SHA256:
    raise ValueError("The preserved code-repair grader has changed")
GRADER = json.loads(GRADER_PATH.read_text(encoding="utf-8"))
if (
    GRADER["type"] != "score_model"
    or GRADER["name"] != "code_repair_quality"
    or GRADER["model"] != "o3-mini-2025-01-31"
    or GRADER["range"] != [0, 1]
    or GRADER["pass_threshold"] != 0.7
    or GRADER["sampling_params"] != {"seed": 42}
):
    raise ValueError("The score-model grader does not match the supplied contract")
bindings = {
    match
    for message in GRADER["input"]
    for match in re.findall(r"\{\{(item|sample)\.([A-Za-z_][A-Za-z_0-9]*)\}\}", message["content"])
}
if bindings != {*(('item', field) for field in GRADER_FIELDS), ('sample', 'output_text')}:
    raise ValueError("Grader placeholders do not match the dataset and response contracts")
print({"grader_model": GRADER["model"], "range": GRADER["range"], "pass_threshold": 0.7,
       "offline_contract": "validated", "model_calibration": "not run"})

## 4. Define one visible managed RFT recipe

The recipe includes exactly the supplied grader and explicitly selects **Global training** (`trainingType=globalStandard`). Omitting the training type defaults to Standard, which this model rejects. Optional training hyperparameters are omitted so Foundry uses model-specific defaults, as recommended in the [RFT guide](https://learn.microsoft.com/azure/foundry/openai/how-to/reinforcement-fine-tuning#configure-hyperparameters). Do not copy Qwen reasoning effort, batch size, epochs, or grader-deployment assumptions into this preview model's job.

Before remote execution, verify the effective training/evaluation defaults, supported training type, budget, preview/RFT access, region, and quota in your project. The grader's seed is not a training seed. Default settings and local timeouts are **not spending caps**.

The fingerprint covers the model, training type, method, and exact input/grader hashes. The short suffix is deterministic so equivalent jobs can be recovered without creating duplicates.

In [ ]:
METHOD = {"type": "reinforcement", "reinforcement": {"grader": GRADER}}
recipe = {"model": MODEL, "training_type": TRAINING_TYPE, "method": METHOD, "dataset_hashes": dataset_hashes,
          "grader_sha256": GRADER_SHA256}
RECIPE_SHA256 = hashlib.sha256(
    json.dumps(recipe, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()


def verify_prepared_inputs():
    for label, contract in DATASETS.items():
        if hash_file(contract["path"]) != contract["sha256"]:
            raise ValueError(f"{label} bytes changed after preparation; stop before remote operations")
    if hash_file(GRADER_PATH) != GRADER_SHA256:
        raise ValueError("Grader bytes changed after preparation")
    current_recipe = {"model": MODEL, "training_type": TRAINING_TYPE, "method": METHOD, "dataset_hashes": dataset_hashes,
                      "grader_sha256": GRADER_SHA256}
    current_fingerprint = hashlib.sha256(
        json.dumps(current_recipe, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    if current_fingerprint != RECIPE_SHA256:
        raise ValueError("The in-memory recipe changed after preparation; revalidate before continuing")


JOB_SUFFIX = NameFactory.for_demo("mai", run_label=RECIPE_SHA256[:8]).remote("rft", max_length=18)
experiment = ExperimentManifest(
    demo_id=DEMO_ID,
    hypothesis="Rewarding functional, minimal repairs may improve code-repair behavior; independent tests are required.",
    base_model=MODEL,
    dataset_hashes={**dataset_hashes, "grader": GRADER_SHA256},
    hyperparameters={"training_type": TRAINING_TYPE, "requested": "model-specific service defaults"},
)
print({"model": MODEL, "training_type": TRAINING_TYPE, "grader": GRADER["name"], "hyperparameters": "service defaults",
       "file_wait_seconds": FILE_TIMEOUT_SECONDS, "monitor_seconds": MONITOR_TIMEOUT_SECONDS})
print("Offline preparation complete. Stop here for an offline check.")

## 5. Configure private project context and recover local state

Copy [the blank template](../.env.template) to a demo-local `.env`. Set `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_RESOURCE_LOCATION` to your own verified project/region. Authenticate using an approved `DefaultAzureCredential` provider. No keys or runtime identifiers are printed.

MAI supports **Global and Developer training**, not Standard or Data zone. These training types do not guarantee data residency in the project's region. Check [supported models and training types](https://learn.microsoft.com/azure/foundry/fine-tuning/overview#supported-models) and confirm the effective service defaults before uploading private data.

Confirm access to `MAI-Code-1.1-Flash-2026-09-15` **and** the supplied `o3-mini-2025-01-31` grader. The service may host the grader or require a separately provisioned deployment; check [model-grader hosting](https://learn.microsoft.com/azure/foundry/openai/how-to/reinforcement-fine-tuning#model-grader-deployments) for this training model. This tutorial does not provision a grader deployment or claim that it is unnecessary.

Review data provenance, license/privacy constraints, upload destination, the model's effective defaults, training/grading costs, and cleanup ownership before the next cells. Pricing, access, roles, and quota remain customer checks; offline validation does not establish them.

`outputs/submission-state.json` is private recovery evidence, not a file to paste into an issue. A different project, region, or recipe is rejected. Use `FOUNDRY_EXISTING_CODE_REPAIR_RFT_JOB_ID` only for a confirmed matching job after reconciling an ambiguous submission.

In [ ]:
config = load_foundry_config(DEMO_ROOT / ".env")
region = os.getenv("FOUNDRY_RESOURCE_LOCATION", "").strip()
if not region:
    raise ValueError("Set FOUNDRY_RESOURCE_LOCATION to the verified training region")
expected_context = {
    "project_endpoint": config.project_endpoint,
    "region": region,
    "training_type": TRAINING_TYPE,
    "recipe_sha256": RECIPE_SHA256,
    "job_suffix": JOB_SUFFIX,
}
if STATE_PATH.is_file():
    runtime = RuntimeManifest(**json.loads(STATE_PATH.read_text(encoding="utf-8")))
    if runtime.demo_id != DEMO_ID or any(
        runtime.resource_context.get(key) != value for key, value in expected_context.items()
    ):
        raise RuntimeError("Saved state belongs to a different project, region, or recipe; reconcile it first")
    if "job" in runtime.operation_ids and not {"training", "validation"} <= runtime.operation_ids.keys():
        raise RuntimeError("Recorded job state is missing its input file IDs")
else:
    runtime = RuntimeManifest(
        demo_id=DEMO_ID,
        resource_context=expected_context,
        sdk_versions={name: version(name) for name in ("azure-ai-projects", "openai")},
    )
confirmed_job_id = os.getenv("FOUNDRY_EXISTING_CODE_REPAIR_RFT_JOB_ID", "").strip()
saved_job_id = runtime.operation_ids.get("job")
if confirmed_job_id and saved_job_id and confirmed_job_id != saved_job_id:
    raise RuntimeError("The recovery job ID conflicts with the recorded job; reconcile before continuing")
print(config.diagnostics())
print({"saved_files": sorted(key for key in runtime.operation_ids if key in DATASETS),
       "job_recorded": bool(saved_job_id), "recovery_id_supplied": bool(confirmed_job_id)})

## 6. Upload or verify the two content-addressed files

**This cell performs live operations and may incur charges.** The destination is the configured project's fine-tuning Files API. No reference fields are stripped or rewritten: upload the exact validated bytes.

Save each file ID before waiting for processing. Reruns retrieve recorded IDs and verify purpose, size, and SHA-256; they do not silently replace missing or changed inputs. New/reused ownership evidence is recorded privately.

If the project's file quota is full, do not delete shared inputs or results. You may adopt an existing processed fine-tune file only after downloading it and verifying its full SHA-256 against the preserved split. Record the confirmed ID and reused ownership in private state; this cell verifies the bytes again before submission.

An in-flight upload marker without a file ID means the outcome is unknown. Reconcile the content-addressed file in Foundry; record its confirmed ID under `operation_ids.training` or `operation_ids.validation` in the private state. Clear the corresponding `<split>_upload_state` marker only if reconciliation proves no file was created. Do not blindly retry an upload after an ambiguous response.

In [ ]:
verify_prepared_inputs()
for label in DATASETS:
    if label not in runtime.operation_ids and runtime.resource_context.get(f"{label}_upload_state") == "in-flight":
        raise RuntimeError(f"{label} upload outcome is unknown; reconcile the file before retrying")
print("This workflow uses paid cloud resources and may incur charges.")
runtime.write(STATE_PATH)
with create_project_context(config) as context:
    client = context.openai_client
    for label, contract in DATASETS.items():
        marker = f"{label}_upload_state"
        if label not in runtime.operation_ids:
            runtime.resource_context[marker] = "in-flight"
            runtime.write(STATE_PATH)
            result = upload_or_reuse_file(
                client, demo_slug=DEMO_ID, purpose_name=label,
                path=contract["path"], wait=False, reuse=True,
            )
            runtime.operation_ids[label] = result.resource_id
            runtime.resource_context[marker] = "recorded"
            runtime.resource_context[f"{label}_file_reused"] = str(result.reused).lower()
            runtime.write(STATE_PATH)
        file_id = runtime.operation_ids[label]
        verified_key = f"{label}_verified_sha256"
        runtime.resource_context.pop(verified_key, None)
        runtime.write(STATE_PATH)
        remote_file = wait_for_file(client, file_id, timeout_seconds=FILE_TIMEOUT_SECONDS)
        if remote_file.purpose != "fine-tune" or remote_file.bytes != contract["bytes"]:
            raise RuntimeError(f"Recorded {label} file purpose or size does not match")
        content = retry_call(lambda: client.files.content(file_id).read(), operation_name="verify uploaded input bytes")
        if hashlib.sha256(content).hexdigest() != contract["sha256"]:
            raise RuntimeError(f"Recorded {label} file bytes do not match the preserved input")
        runtime.resource_context[verified_key] = contract["sha256"]
        runtime.write(STATE_PATH)
        print({"split": label, "file": safe_reference(file_id), "status": remote_file.status, "bytes_verified": True})

## 7. Submit once or recover the exact RFT job

**This cell can start paid training and grading.** It submits `MAI-Code-1.1-Flash-2026-09-15` with `method.type = reinforcement`, both uploaded files, and the unchanged grader. There are no baseline, deployment, tool-execution, or inference calls hidden in this step.

The shared helper disables SDK retries for the POST. We save an in-flight marker before submission, and persist the job ID immediately after a confirmed response. A recorded/recovery ID must match the model, file IDs, grader recipe, suffix, and reusable status; mismatch, failure, or cancellation is an error, not permission to create a replacement.

If the response is lost, check Foundry before rerunning. Set the optional recovery ID only to a confirmed job created by this recipe. If reconciliation proves no job exists, deliberately clear the private `submission_state` marker before an intentional retry. Never erase state just to get past an unknown outcome.

In [ ]:
verify_prepared_inputs()
if not {"training", "validation"} <= runtime.operation_ids.keys():
    raise RuntimeError("Upload and verify both input files before submitting")
if any(runtime.resource_context.get(f"{label}_verified_sha256") != contract["sha256"]
       for label, contract in DATASETS.items()):
    raise RuntimeError("Both remote input files require successful byte verification before submission")
existing_job_id = confirmed_job_id or runtime.operation_ids.get("job")
if runtime.resource_context.get("submission_state") == "in-flight" and not existing_job_id:
    raise RuntimeError("Submission outcome is unknown; reconcile in Foundry and supply the confirmed recovery ID")
if not existing_job_id:
    runtime.resource_context["submission_state"] = "in-flight"
    runtime.write(STATE_PATH)
with create_project_context(config) as context:
    submission = create_or_reuse_fine_tuning_job(
        context.openai_client,
        model=MODEL,
        training_file_id=runtime.operation_ids["training"],
        validation_file_id=runtime.operation_ids["validation"],
        existing_job_id=existing_job_id,
        suffix=JOB_SUFFIX,
        method=METHOD,
        training_type=TRAINING_TYPE,
        reuse=True,
    )
    runtime.operation_ids["job"] = submission.resource_id
    runtime.resource_context["submission_state"] = "recorded"
    runtime.resource_context["job_reused"] = str(submission.reused).lower()
    runtime.resource_context["job_status"] = submission.status
    runtime.write(STATE_PATH)
    experiment.write(OUTPUT_ROOT / "experiment-manifest.json")
    print({"job": safe_reference(submission.resource_id), "status": submission.status, "reused": submission.reused})

## 8. Monitor to a terminal state

Poll every 60 seconds for up to six hours. A timeout stops **local monitoring**, not training, grading, or billing. With this kernel intact, rerun this cell; after kernel loss, rerun preparation and load the saved state before monitoring the recorded job. Do not submit a second job to restart monitoring.

The service-produced fine-tuned model is an artifact identifier, not a deployed inference endpoint. Save it and any result-file IDs privately. Failed or cancelled jobs are recorded and surfaced explicitly. This cell does not cancel or delete anything.

In [ ]:
if "job" not in runtime.operation_ids:
    raise RuntimeError("No recorded job to monitor")
with create_project_context(config) as context:
    completed_job = monitor_fine_tuning_job(
        context.openai_client, runtime.operation_ids["job"],
        poll_interval_seconds=POLL_INTERVAL_SECONDS,
        timeout_seconds=MONITOR_TIMEOUT_SECONDS,
        require_success=False,
    )
    runtime.resource_context["job_status"] = completed_job.status
    runtime.completed_at_utc = datetime.now(timezone.utc).isoformat()
    runtime.write(STATE_PATH)
    if completed_job.status != "succeeded":
        details = redact_text(str(completed_job.error), sensitive_values=(config.project_endpoint,))
        raise RuntimeError(f"RFT ended in {completed_job.status}: {details}")
    if not completed_job.fine_tuned_model:
        raise RuntimeError("The succeeded job has not reported a fine-tuned artifact; inspect service state")
    runtime.operation_ids["fine_tuned_model"] = completed_job.fine_tuned_model
    for index, file_id in enumerate(completed_job.result_files, start=1):
        runtime.operation_ids[f"result_file_{index}"] = file_id
    runtime.write(STATE_PATH)
    print({"status": completed_job.status, "artifact": safe_reference(completed_job.fine_tuned_model),
           "result_files": len(completed_job.result_files)})

## 9. Download and summarize service training metrics

Download the current job's result CSVs into ignored `outputs/`. Summarize finite numeric reward/loss observations using the columns actually returned by the service. Missing metrics, invalid numbers, and unexpected result formats are errors, not zero-valued success reports.

A rising validation reward may mean the policy is better at satisfying this model grader on these 15 validation examples. It does not establish executable correctness, regression safety, or improvement on unseen repair tasks. No historical or fabricated reward curve is substituted for your run.

In [ ]:
def summarize_metrics(content):
    reader = csv.DictReader(io.StringIO(content))
    columns = [name for name in (reader.fieldnames or [])
               if "reward" in name.casefold() or "loss" in name.casefold()]
    if not columns:
        raise ValueError("Service result CSV contains no reward or loss columns")
    rows = list(reader)
    metrics = {}
    for column in columns:
        values = []
        for row in rows:
            raw = row.get(column)
            if raw is None or not raw.strip():
                continue
            value = float(raw)
            if not math.isfinite(value):
                raise ValueError(f"Service metric {column} is not finite")
            values.append(value)
        if values:
            metrics[column] = {"observations": len(values), "first": values[0], "last": values[-1],
                               "min": min(values), "max": max(values)}
    if not metrics:
        raise ValueError("Service result CSV contains no numeric reward or loss observations")
    return {"rows": len(rows), "metrics": metrics}


if runtime.resource_context.get("job_status") != "succeeded":
    raise RuntimeError("Complete monitoring successfully before downloading metrics")
metrics_by_file = {}
with create_project_context(config) as context:
    job = retry_call(lambda: context.openai_client.fine_tuning.jobs.retrieve(runtime.operation_ids["job"]),
                     operation_name="retrieve completed job metrics")
    if job.status != "succeeded" or job.fine_tuned_model != runtime.operation_ids["fine_tuned_model"]:
        raise RuntimeError("Service job no longer matches the recorded completed artifact")
    if not job.result_files:
        raise RuntimeError("The completed job has no result files; inspect service state")
    for index, file_id in enumerate(job.result_files, start=1):
        content = retry_call(lambda: context.openai_client.files.content(file_id).read(),
                             operation_name="download service training metrics")
        label = f"service-result-{index}"
        (OUTPUT_ROOT / f"{label}.csv").write_bytes(content)
        metrics_by_file[label] = summarize_metrics(content.decode("utf-8-sig"))
    service_method = job.model_dump(mode="json").get("method")
    effective = (service_method or {}).get("reinforcement", {}).get("hyperparameters")
    experiment.hyperparameters = {"training_type": TRAINING_TYPE, "requested": "model-specific service defaults",
                                  "service_reported": effective is not None, "effective": effective}
experiment.metrics = metrics_by_file
experiment.write(OUTPUT_ROOT / "experiment-manifest.json")
print(json.dumps(metrics_by_file, indent=2))
print({"effective_hyperparameters_reported": effective is not None})

## 10. Training-only conclusion and lifecycle handoff

You have separated four different outcomes: local data/grader-contract validity, model-grader calibration, training completion, and real code-repair quality. This notebook demonstrates the first and provides the workflow for the third. It does **not** run calibration or independent endpoint evaluation.

**Artifacts:** preserve `outputs/submission-state.json`, `outputs/experiment-manifest.json`, service CSVs, and the recorded fine-tuned artifact. Do not print or commit private state, endpoints, or resource IDs. A registered artifact is not a serving endpoint.

**Remaining resources and costs:** a job that is still queued/running after interruption may continue consuming training/grading resources. Files can be shared by reused jobs. Any separately provisioned grader deployment remains its owner's responsibility. Client context managers close local clients only; they do not cancel jobs, unload remote resources, or delete deployments.

For separately authorized held-out evaluation, serving, cancellation, and deletion, follow [post-training guidance](../../../POST_TRAINING.md). Use an independent test split, isolated executable patch tests, and human review before making quality claims. Never execute generated code in your development environment merely to grade it.

Continue to [agentic RFT](../../../04-agentic-rft/01-agentic-rft-tools/README.md) when the task requires tool-call behavior rather than only a code-repair response.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.